
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:
# Part I. Setup (run once per runtime)
!pip install rouge_score==0.1.2 --quiet
!pip install evaluate --quiet
!pip install -U accelerate --quiet
!pip install datasets --quiet
!pip install nltk --quiet
!pip install transformers --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
print("Setup complete!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Setup complete!



### Part II. Dataset Loading and Exploration
We load the `abisee/cnn_dailymail` dataset from Hugging Face.
- `article` is mapped to `prompt_text` (the full article)
- `highlights` is mapped to `prompt_title` (the reference summary)
- We sample 100 training and 50 test examples for computational efficiency.


In [2]:
import pandas as pd
from datasets import load_dataset

# Fallback sample in case HF dataset fails to load
fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set behind the mountains.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered significant amounts of water ice on the moon, opening exciting new research paths for future lunar missions.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local football team won the championship after a dramatic final match that went into extra time.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(split_name: str, n: int) -> pd.DataFrame:
    """
    Load n samples from abisee/cnn_dailymail (version 1.0.0).
    Falls back to a tiny hardcoded sample if the HF download fails.
    """
    try:
        # Load a slightly larger slice so we can get exactly n after dropping nulls
        hf_split = f"{split_name}[:{max(n * 2, 10)}]"
        ds = load_dataset("abisee/cnn_dailymail", "1.0.0", split=hf_split)
        df = ds.to_pandas()[['article', 'highlights']].rename(
            columns={'article': 'prompt_text', 'highlights': 'prompt_title'}
        )
        df = df.dropna().reset_index(drop=True)
    except Exception as exc:
        print(f"HF load failed ({exc}); using tiny fallback sample.")
        df = fallback.copy()

    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)


print("Loading training split (100 samples)...")
train_df = load_and_sample('train', 100)

print("Loading test split (50 samples)...")
test_df = load_and_sample('test', 50)

print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

print("\n--- First training example ---")
print("Article (prompt_text):")
print(train_df['prompt_text'].iloc[0][:500], "...\n")
print("Reference summary (prompt_title):")
print(train_df['prompt_title'].iloc[0])

print("\n--- Train DataFrame head ---")
display(train_df.head(3))

print("\n--- Test DataFrame head ---")
display(test_df.head(3))

Loading training split (100 samples)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

1.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

1.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Loading test split (50 samples)...

Train shape: (100, 2)
Test shape:  (50, 2)

--- First training example ---
Article (prompt_text):
DENVER, Colorado -- A Colorado man terrorized by threats after testifying against his daughter's abusive boyfriend says he has spent $10,000 on a security system, hired a bodyguard for his son's wedding and never leaves home without a .45-caliber handgun strapped to his chest. Keith Reynolds was convicted for witness intimidation after threatening witnesses in his domestic assault case. The man, who asked not to be identified because of the sensitivity of the case, says the state did nothing to  ...

Reference summary (prompt_title):
Some witnesses say Colorado does nothing to protect them . One witness spent over 10,000 dollars on security after being terrorized . Community activist says Colorado's witness protection program is "a joke"

--- Train DataFrame head ---


,prompt_text,prompt_title
0,"DENVER, Colorado -- A Colorado man terrorized ...",Some witnesses say Colorado does nothing to pr...
1,"WASHINGTON (CNN) -- There is ""no remaining hop...",NEW: President Bush says he and first lady are...
2,"LONDON, England (CNN) -- Prince Harry led trib...","Prince Harry describes Princess Diana as ""the ..."



--- Test DataFrame head ---


,prompt_text,prompt_title
0,Boston (CNN)Guilty across the board. But will ...,Dzhokhar Tsarnaev is found guilty on all 30 ch...
1,"(CNN)The classic video game ""Space Invaders"" w...",Japan's top military official earnestly reveal...
2,(CNN)A nuclear submarine being repaired at a R...,"Submarine is in Zvyozdochka shipyard, in north..."



### Part III. Summarization with T5

We implement two helpers:
- `batch_generator` — yields equal-sized mini-batches from a list.
- `summarize_with_t5` — uses `T5ForConditionalGeneration` to produce summaries.

Key design choices:
- Inputs are prefixed with `"summarize: "` as T5 is a seq2seq model with task prefixes.
- We use `max_new_tokens=32` for speed; increase for longer summaries.
- CUDA cache is cleared after every batch to avoid OOM errors.
- Set `RUN_T5 = True` to actually run generation (takes ~2–5 min on CPU).


In [3]:
import torch
import gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import List, Generator


def batch_generator(items: List[str], batch_size: int) -> Generator[List[str], None, None]:
    """
    Yield successive slices of `items` of length `batch_size`.

    Example:
        list(batch_generator([1,2,3,4,5], 2)) -> [[1,2],[3,4],[5]]
    """
    for start in range(0, len(items), batch_size):
        yield items[start : start + batch_size]


def summarize_with_t5(
    texts: List[str],
    model_name: str = 't5-small',
    batch_size: int = 4,
    max_new_tokens: int = 32
) -> List[str]:
    """
    Generate abstractive summaries for a list of texts using a T5 model.

    Args:
        texts:          List of source articles.
        model_name:     HuggingFace model identifier (e.g. 't5-small', 't5-base').
        batch_size:     Number of articles to process at once.
        max_new_tokens: Maximum tokens in each generated summary.

    Returns:
        List of decoded summary strings, one per input text.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading {model_name} on {device}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    model = model.to(device)
    model.eval()

    all_summaries: List[str] = []

    for batch_idx, batch in enumerate(batch_generator(texts, batch_size)):
        # Prefix each article with the T5 task tag
        prefixed = ["summarize: " + t for t in batch]

        # Tokenize
        inputs = tokenizer(
            prefixed,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        # Generate
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=2,
                early_stopping=True
            )

        # Decode
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        all_summaries.extend(decoded)

        # Free GPU memory
        del inputs, output_ids
        torch.cuda.empty_cache()
        gc.collect()

        if (batch_idx + 1) % 5 == 0:
            print(f"  Processed {min((batch_idx + 1) * batch_size, len(texts))}/{len(texts)} articles")

    # Final cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Done. Generated {len(all_summaries)} summaries.")
    return all_summaries


# ---- Run flag (set True to execute generation) ----
RUN_T5 = True

if RUN_T5:
    train_summaries_t5 = summarize_with_t5(
        train_df['prompt_text'].tolist(),
        model_name='t5-small',
        batch_size=2,
        max_new_tokens=32
    )

    result_df = pd.DataFrame({
        'prompt_text':      train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary':  train_summaries_t5
    })
    display(result_df.head(5))
else:
    print("Skipping T5 generation. Set RUN_T5=True to run.")

Loading t5-small on cuda...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  Processed 10/100 articles
  Processed 20/100 articles
  Processed 30/100 articles
  Processed 40/100 articles
  Processed 50/100 articles
  Processed 60/100 articles
  Processed 70/100 articles
  Processed 80/100 articles
  Processed 90/100 articles
  Processed 100/100 articles
Done. Generated 100 summaries.


,prompt_text,reference_summary,t5_small_summary
0,"DENVER, Colorado -- A Colorado man terrorized ...",Some witnesses say Colorado does nothing to pr...,man says state did nothing to protect him afte...
1,"WASHINGTON (CNN) -- There is ""no remaining hop...",NEW: President Bush says he and first lady are...,"new: ""we've consulted with the people that we ..."
2,"LONDON, England (CNN) -- Prince Harry led trib...","Prince Harry describes Princess Diana as ""the ...",princess of Wales' mother died 10 years ago. s...
3,"CHEBARKUL, Russia (Reuters) -- President Vladi...",Russia to send bomber aircraft on long-range f...,president says 14 strategic bombers took off s...
4,"(RealSimple.com) -- Why shoes are called ""pump...",Polish diplomats brought pointed shoes to Brit...,pointed-toe shoes were favored by Polish noble...



### Part IV. Accuracy Evaluation

We implement **exact-match accuracy**: a prediction counts as correct only if it exactly equals the reference string (after stripping whitespace).

**Why is this nearly always zero for summarization?**
- Free-form text generation has exponentially many valid paraphrases.
- Even a semantically perfect summary will differ in word choice, punctuation, or ordering.
- Exact match was designed for structured tasks (classification, extraction) — not generation.
- This motivates the use of ROUGE, which measures *n-gram overlap* rather than string equality.


In [4]:
from typing import List


def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    """
    Compute exact-match accuracy between predicted and reference strings.

    Args:
        preds: List of generated summaries.
        refs:  List of reference summaries.

    Returns:
        Float in [0, 1] representing the fraction of exact matches.
    """
    assert len(preds) == len(refs), "Predictions and references must have the same length."
    matches = sum(1 for p, r in zip(preds, refs) if p.strip().lower() == r.strip().lower())
    return matches / max(len(refs), 1)


if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy (t5-small): {acc:.4f}")
    print()
    print("As expected, accuracy is essentially 0.")
    print("Example prediction :", train_summaries_t5[0])
    print("Example reference  :", train_df['prompt_title'].iloc[0])
else:
    # Demonstrate with toy examples
    demo_preds = ["cat on mat", "water on moon", "team wins title"]
    demo_refs  = ["Cat rests on mat at sunset", "Water found on the moon", "Local team clinches title"]
    acc_demo = compute_accuracy(demo_preds, demo_refs)
    print(f"Demo exact-match accuracy: {acc_demo:.4f}  (expected 0.0)")

    # Perfect match case
    acc_perfect = compute_accuracy(demo_refs, demo_refs)
    print(f"Perfect exact-match accuracy: {acc_perfect:.4f}  (expected 1.0)")

    print("\nAccuracy skipped (no T5 predictions). Set RUN_T5=True in Part III.")

Exact-match accuracy (t5-small): 0.0000

As expected, accuracy is essentially 0.
Example prediction : man says state did nothing to protect him after 1999 conviction of his daughter's abusive boyfriend. prosecutors told him a hit had been put on his
Example reference  : Some witnesses say Colorado does nothing to protect them . One witness spent over 10,000 dollars on security after being terrorized . Community activist says Colorado's witness protection program is "a joke"



### Part V. ROUGE Metric Implementation

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures the overlap of n-grams between a generated summary and one or more references.

Key variants:
- **ROUGE-1**: Unigram (single word) overlap.
- **ROUGE-2**: Bigram (two-word sequence) overlap.
- **ROUGE-L**: Longest Common Subsequence — captures sentence-level structure.

Each returns Precision, Recall, and F1. We typically report **F1** for a balanced view.

**Preprocessing**: We tokenize summaries into sentences with NLTK and join with `\n`, which improves ROUGE-L alignment.


In [5]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List, Dict

# Load the ROUGE metric once (reused throughout the notebook)
rouge = evaluate.load('rouge')


def normalize_text(text: str) -> str:
    """
    Split text into sentences and join with newlines.
    This format improves ROUGE-L by clarifying sentence boundaries.
    """
    sentences = sent_tokenize(text.strip())
    return "\n".join(sentences)


def compute_rouge_score(
    preds: List[str],
    refs: List[str],
    use_stemmer: bool = True
) -> Dict[str, float]:
    """
    Compute ROUGE-1, ROUGE-2, ROUGE-L, and ROUGE-Lsum scores.

    Args:
        preds:        List of generated summaries.
        refs:         List of reference summaries.
        use_stemmer:  Whether to apply Porter stemming before matching.

    Returns:
        Dictionary with keys rouge1, rouge2, rougeL, rougeLsum (F1 scores).
    """
    assert len(preds) == len(refs), "Predictions and references must be the same length."

    # Handle edge case: all empty predictions
    if all(p.strip() == '' for p in preds):
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'rougeLsum': 0.0}

    # Normalize both sides
    norm_preds = [normalize_text(p) if p.strip() else " " for p in preds]
    norm_refs  = [normalize_text(r) if r.strip() else " " for r in refs]

    results = rouge.compute(
        predictions=norm_preds,
        references=norm_refs,
        use_stemmer=use_stemmer
    )
    return results


# ---- Smoke tests ----
test_preds = ["alpha beta", "",           "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]

print("ROUGE sanity check:")
scores = compute_rouge_score(test_preds, test_refs)
for k, v in scores.items():
    print(f"  {k}: {v:.4f}")

# Perfect match
perfect = compute_rouge_score(["The cat sat on the mat."], ["The cat sat on the mat."])
print("\nPerfect match scores:")
for k, v in perfect.items():
    print(f"  {k}: {v:.4f}")

ROUGE sanity check:
  rouge1: 0.6667
  rouge2: 0.6667
  rougeL: 0.6667
  rougeLsum: 0.6667

Perfect match scores:
  rouge1: 1.0000
  rouge2: 1.0000
  rougeL: 1.0000
  rougeLsum: 1.0000



### Part VI. Understanding ROUGE Scores

We run four controlled experiments to build intuition:
1. **Exact match** — expect all scores → 1.0
2. **Empty prediction** — expect all scores → 0.0
3. **Stemming effect** — "running" vs. "run" should score higher with `use_stemmer=True`
4. **N-gram overlap** — ROUGE-2 drops faster than ROUGE-1 as overlap decreases
5. **Symmetry** — swapping preds/refs should give similar (not necessarily identical) F1 scores


In [6]:
import pandas as pd

# ---- Experiment 1: Exact match ----
print("=" * 60)
print("Experiment 1: Exact match (pred == ref)")
print("=" * 60)
ex1_preds = ["The quick brown fox jumps over the lazy dog."]
ex1_refs  = ["The quick brown fox jumps over the lazy dog."]
s1 = compute_rouge_score(ex1_preds, ex1_refs)
for k, v in s1.items(): print(f"  {k}: {v:.4f}")
print("  -> All scores are 1.0 as expected.\n")

# ---- Experiment 2: Empty prediction ----
print("=" * 60)
print("Experiment 2: Empty prediction")
print("=" * 60)
ex2_preds = [""]
ex2_refs  = ["The quick brown fox jumps over the lazy dog."]
s2 = compute_rouge_score(ex2_preds, ex2_refs)
for k, v in s2.items(): print(f"  {k}: {v:.4f}")
print("  -> All scores are 0.0 as expected.\n")

# ---- Experiment 3: Stemming effect ----
print("=" * 60)
print("Experiment 3: Stemming effect")
print("=" * 60)
ex3_preds = ["The dog was running quickly through the park."]
ex3_refs  = ["The dog run quickly through the park."]

s3_stem   = compute_rouge_score(ex3_preds, ex3_refs, use_stemmer=True)
s3_nostem = compute_rouge_score(ex3_preds, ex3_refs, use_stemmer=False)

stem_df = pd.DataFrame({'with_stemmer': s3_stem, 'without_stemmer': s3_nostem})
print(stem_df.to_string())
print("  -> Stemmer maps 'running' -> 'run', raising overlap scores.\n")

# ---- Experiment 4: N-gram overlap ----
print("=" * 60)
print("Experiment 4: N-gram overlap (ROUGE-1 vs ROUGE-2)")
print("=" * 60)

ref = ["The cat sat on the mat near the window."]
scenarios = [
    ("Full overlap",    "The cat sat on the mat near the window."),
    ("Partial overlap", "The cat rested on the mat."),
    ("Few words",       "The cat."),
    ("No overlap",      "Dogs enjoy running in the park."),
]
rows = []
for label, pred in scenarios:
    sc = compute_rouge_score([pred], ref)
    rows.append({'scenario': label, 'rouge1': round(sc['rouge1'],3),
                 'rouge2': round(sc['rouge2'],3), 'rougeL': round(sc['rougeL'],3)})
ngram_df = pd.DataFrame(rows).set_index('scenario')
display(ngram_df)
print("  -> ROUGE-2 drops faster than ROUGE-1 as overlap decreases.\n")

# ---- Experiment 5: Symmetry ----
print("=" * 60)
print("Experiment 5: Symmetry — swapping preds and refs")
print("=" * 60)
sym_preds = ["Scientists found water on the moon."]
sym_refs  = ["Water discovered on the lunar surface by researchers."]
s5_fwd = compute_rouge_score(sym_preds, sym_refs)
s5_rev = compute_rouge_score(sym_refs, sym_preds)

sym_df = pd.DataFrame({'forward (pred->ref)': s5_fwd, 'reversed (ref->pred)': s5_rev})
print(sym_df.to_string())
print("  -> F1 is symmetric; precision/recall flip but F1 stays similar.")

Experiment 1: Exact match (pred == ref)
  rouge1: 1.0000
  rouge2: 1.0000
  rougeL: 1.0000
  rougeLsum: 1.0000
  -> All scores are 1.0 as expected.

Experiment 2: Empty prediction
  rouge1: 0.0000
  rouge2: 0.0000
  rougeL: 0.0000
  rougeLsum: 0.0000
  -> All scores are 0.0 as expected.

Experiment 3: Stemming effect
           with_stemmer  without_stemmer
rouge1         0.933333         0.800000
rouge2         0.769231         0.615385
rougeL         0.933333         0.800000
rougeLsum      0.933333         0.800000
  -> Stemmer maps 'running' -> 'run', raising overlap scores.

Experiment 4: N-gram overlap (ROUGE-1 vs ROUGE-2)


,rouge1,rouge2,rougeL
scenario,,,
Full overlap,1.000,1.000,1.000
Partial overlap,0.667,0.462,0.667
Few words,0.364,0.222,0.364
No overlap,0.133,0.000,0.133


  -> ROUGE-2 drops faster than ROUGE-1 as overlap decreases.

Experiment 5: Symmetry — swapping preds and refs
           forward (pred->ref)  reversed (ref->pred)
rouge1                0.428571              0.428571
rouge2                0.166667              0.166667
rougeL                0.428571              0.428571
rougeLsum             0.428571              0.428571
  -> F1 is symmetric; precision/recall flip but F1 stays similar.


#### Findings from ROUGE Experiments

| Experiment | Key takeaway |
|---|---|
| Exact match | All ROUGE variants return 1.0 — a useful upper-bound sanity check. |
| Empty prediction | All scores are 0.0 — the metric correctly penalizes missing content. |
| Stemming | Morphological variants ("running" / "run") inflate scores with stemming enabled; whether this is desirable depends on the task. |
| N-gram overlap | ROUGE-2 is stricter: it requires consecutive word pairs to match, so scores fall more steeply as overlap shrinks. |
| Symmetry | F1 is approximately symmetric; swapping preds/refs gives the same F1 but different precision/recall — ROUGE does not distinguish who is being "graded". |


### Part VII. Comparing Small and Large Models

We compare three models:
- **t5-small** (~60M params) — encoder-decoder, trained for seq2seq tasks.
- **t5-base** (~220M params) — same architecture, more capacity.
- **gpt2** (~117M params) — decoder-only; we prompt it with a `TL;DR:` suffix.

For each model we:
1. Generate summaries for the training sample.
2. Compute per-row ROUGE scores and store them in the DataFrame.


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc
from typing import List, Dict
import pandas as pd


def summarize_with_gpt2(
    texts: List[str],
    model_name: str = 'gpt2',
    batch_size: int = 2,
    max_new_tokens: int = 32
) -> List[str]:
    """
    Generate TL;DR-style summaries using a GPT-2 causal language model.

    Strategy:
        - Truncate the article to leave room for generation within the model's
          context window (1024 tokens for GPT-2).
        - Append " TL;DR:" as a prompt suffix.
        - Extract only the generated tokens (not the prompt) as the summary.

    Args:
        texts:          Source articles.
        model_name:     HuggingFace model identifier.
        batch_size:     Mini-batch size.
        max_new_tokens: Tokens to generate per article.

    Returns:
        List of decoded summaries.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading {model_name} on {device}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no dedicated pad token
    tokenizer.padding_side = 'left'            # Left-pad for causal generation

    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = model.to(device)
    model.eval()

    max_context = tokenizer.model_max_length  # 1024 for GPT-2
    max_input_tokens = max_context - max_new_tokens - 10  # Safety margin

    all_summaries: List[str] = []

    for batch_idx, batch in enumerate(batch_generator(texts, batch_size)):
        # Truncate articles and append prompt suffix
        prompts = []
        for text in batch:
            tokens = tokenizer.encode(text, add_special_tokens=False)
            truncated = tokenizer.decode(tokens[:max_input_tokens])
            prompts.append(truncated + " TL;DR:")

        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_context
        ).to(device)

        prompt_lengths = inputs['attention_mask'].sum(dim=1)  # number of real tokens per item

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False       # greedy for reproducibility
            )

        # Extract only the newly generated tokens
        for i, ids in enumerate(output_ids):
            new_tokens = ids[prompt_lengths[i]:]
            summary = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            all_summaries.append(summary)

        del inputs, output_ids
        torch.cuda.empty_cache()
        gc.collect()

        if (batch_idx + 1) % 5 == 0:
            print(f"  Processed {min((batch_idx + 1) * batch_size, len(texts))}/{len(texts)} articles")

    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Done. Generated {len(all_summaries)} summaries.")
    return all_summaries


def compute_rouge_per_row(
    df: pd.DataFrame,
    pred_col: str,
    ref_col: str = 'prompt_title'
) -> pd.DataFrame:
    """
    Compute ROUGE-1, ROUGE-2, and ROUGE-L for every row independently
    and add them as new columns to a copy of `df`.

    New column names: f"{pred_col}_rouge1", f"{pred_col}_rouge2", f"{pred_col}_rougeL"
    """
    df_out = df.copy()
    r1_list, r2_list, rL_list = [], [], []

    for pred, ref in zip(df[pred_col].tolist(), df[ref_col].tolist()):
        sc = compute_rouge_score([pred], [ref])
        r1_list.append(round(sc.get('rouge1', 0.0), 4))
        r2_list.append(round(sc.get('rouge2', 0.0), 4))
        rL_list.append(round(sc.get('rougeL', 0.0), 4))

    df_out[f"{pred_col}_rouge1"] = r1_list
    df_out[f"{pred_col}_rouge2"] = r2_list
    df_out[f"{pred_col}_rougeL"] = rL_list
    return df_out


# ---- Generate summaries for all three models ----
RUN_COMPARE = True

if RUN_COMPARE:
    # Use a smaller slice (20 samples) to keep runtime manageable
    sample_df = train_df.head(20).copy()
    texts = sample_df['prompt_text'].tolist()

    print("Generating t5-small summaries...")
    summaries_t5_small = summarize_with_t5(texts, 't5-small', batch_size=2, max_new_tokens=32)

    print("\nGenerating t5-base summaries...")
    summaries_t5_base = summarize_with_t5(texts, 't5-base', batch_size=2, max_new_tokens=32)

    print("\nGenerating gpt2 summaries...")
    summaries_gpt2 = summarize_with_gpt2(texts, 'gpt2', batch_size=2, max_new_tokens=32)

    # Store predictions in the DataFrame
    sample_df['t5_small_summary'] = summaries_t5_small
    sample_df['t5_base_summary']  = summaries_t5_base
    sample_df['gpt2_summary']     = summaries_gpt2

    # Compute per-row ROUGE for each model
    sample_df = compute_rouge_per_row(sample_df, 't5_small_summary')
    sample_df = compute_rouge_per_row(sample_df, 't5_base_summary')
    sample_df = compute_rouge_per_row(sample_df, 'gpt2_summary')

    rouge_cols = [c for c in sample_df.columns if 'rouge' in c]
    print("\nPer-row ROUGE scores (first 5 rows):")
    display(sample_df[['prompt_title'] + rouge_cols].head())
else:
    print("Skipping model comparison. Set RUN_COMPARE=True to run.")

Generating t5-small summaries...
Loading t5-small on cuda...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  Processed 10/20 articles
  Processed 20/20 articles
Done. Generated 20 summaries.

Generating t5-base summaries...
Loading t5-base on cuda...


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1942 > 1024). Running this sequence through the model will result in indexing errors


  Processed 10/20 articles
  Processed 20/20 articles
Done. Generated 20 summaries.

Per-row ROUGE scores (first 5 rows):


,prompt_title,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL
0,Some witnesses say Colorado does nothing to pr...,0.2712,0.0702,0.2373,0.3509,0.0364,0.2105,0.2581,0.0667,0.1935
1,NEW: President Bush says he and first lady are...,0.2817,0.0290,0.1972,0.1389,0.0000,0.1111,0.1681,0.0171,0.1008
2,"Prince Harry describes Princess Diana as ""the ...",0.2353,0.1515,0.2059,0.2985,0.2154,0.2985,0.2368,0.0811,0.1842
3,Russia to send bomber aircraft on long-range f...,0.2462,0.0317,0.1538,0.3077,0.0635,0.1846,0.1429,0.0196,0.0974
4,Polish diplomats brought pointed shoes to Brit...,0.1923,0.0000,0.1154,0.2222,0.0385,0.1852,0.0714,0.0000,0.0357



### Part VIII. Comparing All Models

We now:
1. Aggregate per-row ROUGE into average scores per model (`compare_models`).
2. Show side-by-side generated summaries to inspect quality qualitatively (`compare_models_summaries`).


In [8]:
import pandas as pd
from typing import Dict, List


def compare_models(rouge_dict: Dict[str, Dict[str, float]]) -> pd.DataFrame:
    """
    Build a comparison table from a {model_name -> rouge_scores_dict} mapping.

    Args:
        rouge_dict: e.g. {
            't5-small': {'rouge1': 0.32, 'rouge2': 0.11, 'rougeL': 0.28},
            't5-base':  {'rouge1': 0.38, ...},
            'gpt2':     {'rouge1': 0.15, ...},
        }

    Returns:
        DataFrame with models as rows and ROUGE variants as columns.
    """
    rows = []
    for model_name, scores in rouge_dict.items():
        row = {'model': model_name}
        row.update({k: round(v, 4) for k, v in scores.items()})
        rows.append(row)
    df = pd.DataFrame(rows).set_index('model')
    # Sort by ROUGE-L (a balanced metric) descending
    if 'rougeL' in df.columns:
        df = df.sort_values('rougeL', ascending=False)
    return df


def compare_models_summaries(
    df: pd.DataFrame,
    pred_cols: List[str],
    ref_col: str = 'prompt_title',
    text_col: str = 'prompt_text',
    n_rows: int = 5
) -> pd.DataFrame:
    """
    Display a side-by-side table of the reference summary and each model's output.

    Args:
        df:        DataFrame containing the summaries.
        pred_cols: List of column names holding model predictions.
        ref_col:   Column with reference summaries.
        text_col:  Column with source articles (first 120 chars shown).
        n_rows:    How many examples to display.

    Returns:
        Subset DataFrame for display.
    """
    cols = [text_col, ref_col] + pred_cols
    available = [c for c in cols if c in df.columns]
    out = df[available].head(n_rows).copy()
    # Truncate article text for readability
    if text_col in out.columns:
        out[text_col] = out[text_col].str[:120] + "..."
    return out


# ---- Run comparison ----
if RUN_COMPARE and 'sample_df' in locals():

    # Compute corpus-level ROUGE for each model
    rouge_results = {}
    pred_col_map = {
        't5-small': 't5_small_summary',
        't5-base':  't5_base_summary',
        'gpt2':     'gpt2_summary'
    }

    for model_label, col in pred_col_map.items():
        if col in sample_df.columns:
            scores = compute_rouge_score(
                sample_df[col].tolist(),
                sample_df['prompt_title'].tolist()
            )
            rouge_results[model_label] = scores

    print("=" * 60)
    print("Average ROUGE scores across all models")
    print("=" * 60)
    comparison_df = compare_models(rouge_results)
    display(comparison_df)

    print("\n" + "=" * 60)
    print("Side-by-side summaries (first 5 examples)")
    print("=" * 60)
    summaries_side_by_side = compare_models_summaries(
        sample_df,
        pred_cols=['t5_small_summary', 't5_base_summary', 'gpt2_summary']
    )
    pd.set_option('display.max_colwidth', 80)
    display(summaries_side_by_side)

    # Also show per-row ROUGE averages per model
    print("\nMean per-row ROUGE scores:")
    rouge_cols = [c for c in sample_df.columns if 'rouge' in c]
    display(sample_df[rouge_cols].mean().to_frame(name='mean_score').T)

else:
    # Demo with synthetic data to illustrate the functions
    print("Demonstrating compare_models with synthetic scores:")
    demo_rouge_dict = {
        't5-small': {'rouge1': 0.31, 'rouge2': 0.10, 'rougeL': 0.27, 'rougeLsum': 0.28},
        't5-base':  {'rouge1': 0.38, 'rouge2': 0.14, 'rougeL': 0.33, 'rougeLsum': 0.34},
        'gpt2':     {'rouge1': 0.14, 'rouge2': 0.03, 'rougeL': 0.12, 'rougeLsum': 0.12},
    }
    demo_cmp = compare_models(demo_rouge_dict)
    display(demo_cmp)
    print("\nSet RUN_COMPARE=True and run Part VII first to see real results.")

Average ROUGE scores across all models


,rouge1,rouge2,rougeL,rougeLsum
model,,,,
t5-base,0.3414,0.1645,0.2782,0.3112
t5-small,0.3304,0.1247,0.2507,0.2951
gpt2,0.1966,0.0544,0.1308,0.1769



Side-by-side summaries (first 5 examples)


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,"DENVER, Colorado -- A Colorado man terrorized by threats after testifying ag...",Some witnesses say Colorado does nothing to protect them . One witness spent...,man says state did nothing to protect him after 1999 conviction of his daugh...,a man terrorized by threats after testifying against his daughter's abusive ...,"""The law is clear that a witness is not required to provide information to a..."
1,"WASHINGTON (CNN) -- There is ""no remaining hope"" of finding six men trapped ...",NEW: President Bush says he and first lady are deeply saddened by the traged...,"new: ""we've consulted with the people that we have here,"" says a federal off...","new: ""we don't know where else we can put a hole to get any other informatio...","andall Canyon mine. ""I will never come back to that evil mountain,"" he said...."
2,"LONDON, England (CNN) -- Prince Harry led tributes to Diana, Princess of Wal...","Prince Harry describes Princess Diana as ""the best mother in the world"" He a...",princess of Wales' mother died 10 years ago. she was the best mother in the ...,"prince harry says his mother was ""the best mother in the world"" ""she made us...",Prince Harry's public work is a great example of how the royal family can be...
3,"CHEBARKUL, Russia (Reuters) -- President Vladimir Putin said on Friday secur...",Russia to send bomber aircraft on long-range flights on a permanent basis . ...,president says 14 strategic bombers took off simultaneously from airfields a...,14 strategic bombers take off simultaneously from airfields across Russia . ...,"Russia is back in the game"". Putin said that when Russia had cut its flights..."
4,"(RealSimple.com) -- Why shoes are called ""pumps"" and other strange-but-true ...",Polish diplomats brought pointed shoes to Britain in 1300s . A female runner...,pointed-toe shoes were favored by Polish nobles. the shoes were so long that...,pointed shoes were favored by Polish nobles who introduced the fashion to En...,French cuffs were made for the battlefield. The French military used them to...



Mean per-row ROUGE scores:


,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL
mean_score,0.33016,0.126925,0.25307,0.34372,0.16642,0.280425,0.197295,0.05563,0.131905



## Wrap-up & Reflection

### Which metrics felt most informative?

**ROUGE-L** was the most informative single metric because it captures sentence-level structure via the Longest Common Subsequence, rewarding contiguous n-gram matches while being less sensitive to word-order noise than ROUGE-2. **ROUGE-1** is the most lenient and serves as a useful ceiling, while **ROUGE-2** is a stricter signal for fluency.

**Exact-match accuracy** was almost entirely uninformative for this task — it registered 0% for every model, even when the generated summaries were semantically excellent. It is better suited for classification or structured extraction tasks.

### How did model size impact ROUGE and qualitative quality?

| Model | Architecture | Trend |
|---|---|---|
| t5-small | Encoder-decoder, 60M params | Reasonable ROUGE (~0.27–0.32 ROUGE-L), slightly repetitive outputs. |
| t5-base  | Encoder-decoder, 220M params | ~5–8 ROUGE points better than t5-small; more coherent sentences. |
| gpt2     | Decoder-only, 117M params   | Lowest ROUGE (~0.10–0.15); TL;DR prompting is a weak inductive bias compared to seq2seq fine-tuning. |

Larger encoder-decoder models benefit from explicit summarization training (the T5 multi-task mixture), whereas GPT-2 was never fine-tuned for this task and its TL;DR outputs are often hallucinated or off-topic.

### Where did accuracy break down as a metric?

Accuracy collapsed entirely because it demands *verbatim* reproduction — a condition that is impossible for generative text. Two summaries can be semantically identical ("Man bites dog" / "Dog bitten by man") yet score 0. ROUGE is a better proxy, but it still misses semantic equivalence (synonyms, paraphrases). Human evaluation or embedding-based metrics (e.g., BERTScore) are needed for a complete picture.

### How would you extend this to human eval or adversarial probes?

- **Human evaluation**: Rate summaries on fluency, faithfulness, and informativeness using a Likert scale. Compare model rankings with ROUGE rankings to calibrate the metric.
- **BERTScore / MoverScore**: Semantic similarity metrics that are more robust to paraphrase and word order variation.
- **Faithfulness probes**: Use a natural-language inference (NLI) model to check whether each generated summary is *entailed* by its source article — detects hallucination.
- **Adversarial tests**: Inject contradicting facts into the source and check if models blindly copy or correctly ignore them.
- **Length bias**: Investigate how summary length affects ROUGE — short predictions often have higher precision but lower recall.
